In [1]:
import pandas as pd

In [2]:
ecolo = pd.read_csv('eval_data/ECOLO.csv')
ecolo_screened = pd.read_csv('eval_data/ECOLO_screening_results.csv')
socio = pd.read_csv('eval_data/SOCIO.csv')
socio_screened = pd.read_csv('eval_data/SOCIO_screening_results.csv')
print(len(ecolo), len(ecolo_screened), len(socio), len(socio_screened))

32473 6135 49421 13219


In [3]:
ecolo = ecolo[ecolo.abstract.notna()]
ecolo_screened = ecolo_screened[ecolo_screened.abstract.notna()]
socio = socio[socio.abstract.notna()]
socio_screened = socio_screened[socio_screened.abstract.notna()]
print(len(ecolo), len(ecolo_screened), len(socio), len(socio_screened))

32150 6091 48726 13000


In [4]:
# drop duplicates
ecolo = ecolo.drop_duplicates(subset=['abstract'])
ecolo_screened = ecolo_screened.drop_duplicates(subset=['abstract'])
socio = socio.drop_duplicates(subset=['abstract'])
socio_screened = socio_screened.drop_duplicates(subset=['abstract'])
print(len(ecolo), len(ecolo_screened), len(socio), len(socio_screened))

32091 6089 48650 12998


In [5]:
# keep only when doi present so we can join by it (record_id is not reliable)
ecolo = ecolo[ecolo.doi.notna()]
ecolo_screened = ecolo_screened[ecolo_screened.doi.notna()]
socio = socio[socio.doi.notna()]
socio_screened = socio_screened[socio_screened.doi.notna()]
print(len(ecolo), len(ecolo_screened), len(socio), len(socio_screened))

27166 5486 41136 11274


In [ ]:
ecolo['kept'] = ecolo.doi.isin(ecolo_screened.doi)
socio['kept'] = socio.doi.isin(socio_screened.doi)

In [8]:
ecolo['origin'] = 'ecolo'
socio['origin'] = 'socio'

In [9]:
df = pd.concat([
    ecolo[['doi', 'origin', 'primary_title', 'abstract', 'kept']],
    socio[['doi', 'origin', 'primary_title', 'abstract', 'kept']],
], ignore_index=True)
print(df.kept.mean())
df

0.2481332903868115


,doi,origin,primary_title,abstract,kept
0,10.1016/j.energy.2014.12.003,ecolo,The influence of price and non-price effects o...,This paper models energy demand for space and ...,False
1,10.1088/1755-1315/214/1/012138,ecolo,Possibilities of Liquefied Natural Gas (LNG) u...,Liquefied natural gas (LNG) has an important r...,False
2,10.2478/oszn-2023-0016,ecolo,Accuracy of the Copernicus High-Resolution Lay...,"Over the past years, several remote sensing ma...",False
3,10.1016/j.enbuild.2015.03.018,ecolo,A transdisciplinary approach on the energy eff...,Abstract Buildings account for 40% of total en...,False
4,10.1002/er.3636,ecolo,Experimental investigation on flow improvement...,There is no general rule in the literature to ...,False
...,...,...,...,...,...
68297,10.5593/sgem2019/4.1/S19.120,socio,Design of pavement structures in tunnels,The transportation infrastructure belongs amon...,False
68298,10.31952/amha.17.2.6,socio,ZVONIMIR MARETIC (1921-1989): THE HISTORY OF D...,Zvonimir Maretic was the pioneer of the study ...,False
68299,10.1007/s12061-016-9189-z,socio,The Path-Dependency of Low-Income Neighbourhoo...,The gap between wealthy and disadvantaged neig...,True
68300,10.5593/sgem2022/5.1/s21.075,socio,GREEN COMPETITIVENESS IN WASTE MANAGEMENT IN U...,"Waste management is a public service, the prov...",False


In [10]:
df.origin.value_counts()

origin
socio    41136
ecolo    27166
Name: count, dtype: int64

In [11]:
from setfit import SetFitModel

In [12]:
model = SetFitModel.from_pretrained("TheoLvs/wsl-prescreening-multi-v0.0")

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config_setfit.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model_head.pkl:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

In [13]:
def get_preds(df):
  preds = model.predict_proba(df['abstract'].to_list(), batch_size=100, show_progress_bar=True)
  labelnames = ["other","planetary_boundaries","well_being","resources","justice"]
  cols = ["proba_"+x for x in labelnames]
  preds_df = pd.DataFrame(preds[:,:,1].numpy(), columns=cols)
  return pd.concat((df, preds_df), axis=1)

In [14]:
preds = get_preds(df)

Batches:   0%|          | 0/684 [00:00<?, ?it/s]

In [35]:
preds

,doi,origin,primary_title,abstract,kept,proba_other,proba_planetary_boundaries,proba_well_being,proba_resources,proba_justice,prescreening_high,prescreening_medium,prescreening_low,pred_class
0,10.1016/j.energy.2014.12.003,ecolo,The influence of price and non-price effects o...,This paper models energy demand for space and ...,False,0.021045,0.708858,0.030877,0.241656,0.027327,True,True,False,planetary_boundaries
1,10.1088/1755-1315/214/1/012138,ecolo,Possibilities of Liquefied Natural Gas (LNG) u...,Liquefied natural gas (LNG) has an important r...,False,0.037535,0.820993,0.039448,0.048964,0.040621,True,True,True,planetary_boundaries
2,10.2478/oszn-2023-0016,ecolo,Accuracy of the Copernicus High-Resolution Lay...,"Over the past years, several remote sensing ma...",False,0.041512,0.823406,0.040466,0.041268,0.042024,True,True,True,planetary_boundaries
3,10.1016/j.enbuild.2015.03.018,ecolo,A transdisciplinary approach on the energy eff...,Abstract Buildings account for 40% of total en...,False,0.017278,0.654689,0.049041,0.248701,0.024977,True,True,False,planetary_boundaries
4,10.1002/er.3636,ecolo,Experimental investigation on flow improvement...,There is no general rule in the literature to ...,False,0.049575,0.817899,0.043036,0.035180,0.040137,True,True,True,planetary_boundaries
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68297,10.5593/sgem2019/4.1/S19.120,socio,Design of pavement structures in tunnels,The transportation infrastructure belongs amon...,False,0.040174,0.817663,0.038827,0.049289,0.039071,True,True,True,planetary_boundaries
68298,10.31952/amha.17.2.6,socio,ZVONIMIR MARETIC (1921-1989): THE HISTORY OF D...,Zvonimir Maretic was the pioneer of the study ...,False,0.852792,0.033384,0.037812,0.037695,0.038630,False,False,False,other
68299,10.1007/s12061-016-9189-z,socio,The Path-Dependency of Low-Income Neighbourhoo...,The gap between wealthy and disadvantaged neig...,True,0.838254,0.051467,0.045282,0.029507,0.030212,False,False,False,other
68300,10.5593/sgem2022/5.1/s21.075,socio,GREEN COMPETITIVENESS IN WASTE MANAGEMENT IN U...,"Waste management is a public service, the prov...",False,0.045637,0.819455,0.040649,0.039601,0.040626,True,True,True,planetary_boundaries


In [16]:
save = preds.copy()

In [17]:
def add_screen_class(df):
    preds_df = df[
        [
            "proba_other",
            "proba_planetary_boundaries",
            "proba_well_being",
            "proba_resources",
            "proba_justice",
        ]
    ]
    prescreening_high = preds_df.idxmax(axis=1) != "proba_other"
    pred_class = preds_df.idxmax(axis=1).map(lambda x: x.replace("proba_", ""))

    prescreening_medium = prescreening_high.copy()
    prescreening_medium.loc[preds_df.max(axis=1) < 0.5] = False

    prescreening_low = prescreening_high.copy()
    prescreening_low.loc[preds_df.max(axis=1) < 0.8] = False

    df["prescreening_high"] = prescreening_high
    df["prescreening_medium"] = prescreening_medium
    df["prescreening_low"] = prescreening_low
    df["pred_class"] = pred_class

    return df

In [19]:
preds = add_screen_class(preds)

In [25]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

In [36]:
print("Kept mean:", preds.kept.mean())
print()
print("High threshold")
print("Mean:", preds.prescreening_high.mean())
print("Precision:", precision_score(preds.kept, preds.prescreening_high))
print("Recall:", recall_score(preds.kept, preds.prescreening_high))
print("F1 Score:", f1_score(preds.kept, preds.prescreening_high))
print()
print("Medium threshold")
print("Mean:", preds.prescreening_medium.mean())
print("Precision:", precision_score(preds.kept, preds.prescreening_medium))
print("Recall:", recall_score(preds.kept, preds.prescreening_medium))
print("F1 Score:", f1_score(preds.kept, preds.prescreening_medium))
print()
print("Low threshold")
print("Mean:", preds.prescreening_low.mean())
print("Precision:", precision_score(preds.kept, preds.prescreening_low))
print("Recall:", recall_score(preds.kept, preds.prescreening_low))
print("F1 Score:", f1_score(preds.kept, preds.prescreening_low))

Kept mean: 0.2481332903868115

High threshold
Mean: 0.7474158882609587
Precision: 0.2569637610186092
Recall: 0.7740146329950437
F1 Score: 0.38583487749639694

Medium threshold
Mean: 0.7040789435155632
Precision: 0.24763984196298608
Recall: 0.7026787821571867
F1 Score: 0.3662166733294382

Low threshold
Mean: 0.34479224620069693
Precision: 0.20980891719745223
Recall: 0.2915388246400755
F1 Score: 0.24401204997777667
